In [1]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
train_df=pd.read_csv('/kaggle/input/titanic/train.csv')
test_df=pd.read_csv('/kaggle/input/titanic/test.csv')

In [3]:
# Display all columns without truncation
# pd.set_option('display.max_columns', None)

# Display all output in a single wide row without wrapping
# pd.set_option('display.width', 1000)

In [4]:
# Display the first 5 rows of the dataset
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Check data types and missing values
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
# Calculate missing values count and percentage
missing_count = train_df.isnull().sum()
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100

# Combine results into a DataFrame
missing_data = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percent
})

# Filter and show only columns with missing values
missing_data = missing_data[missing_data['Missing Count'] > 0]
print(missing_data)

          Missing Count  Percentage (%)
Age                 177       19.865320
Cabin               687       77.104377
Embarked              2        0.224467


In [7]:
# Calculate missing values count and percentage
missing_count = test_df.isnull().sum()
missing_percent = (test_df.isnull().sum() / len(test_df)) * 100

# Combine results into a DataFrame
missing_data = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percent
})

# Filter and show only columns with missing values
missing_data = missing_data[missing_data['Missing Count'] > 0]
print(missing_data)

       Missing Count  Percentage (%)
Age               86       20.574163
Fare               1        0.239234
Cabin            327       78.229665


In [8]:
# 1. Fill missing Age with train median
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
test_df['Age'].fillna(train_df['Age'].median(), inplace=True)

# 2. Fill missing Fare in test data with train median
test_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)

# 3. Fill missing Embarked in train data with mode
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

# 4. Drop Cabin column from both dataframes
train_df.drop(columns=['Cabin'], inplace=True)
test_df.drop(columns=['Cabin'], inplace=True)

# 5. Verify missing values are cleared
print("--- Train Missing Values ---")
print(train_df.isnull().sum())
print("\n--- Test Missing Values ---")
print(test_df.isnull().sum())

--- Train Missing Values ---
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

--- Test Missing Values ---
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [9]:
# 2. Save PassengerId from test_df for final submission file
test_passenger_ids = test_df['PassengerId']

# Drop unused columns from both train and test data
cols_to_drop = ['PassengerId', 'Name', 'Ticket']
train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)


In [10]:
# 1. Convert categorical text columns to numerical values
# Sex: male -> 0, female -> 1
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})

# Embarked: S -> 0, C -> 1, Q -> 2
train_df['Embarked'] = train_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test_df['Embarked'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})


In [11]:
# 4. Display the processed train data
print("--- Processed Train Data ---")
train_df.head()

--- Processed Train Data ---


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


In [12]:

# Separate features (X) and target variable (y)
X = train_df.drop(columns=['Survived'])
y = train_df['Survived']

# Test dataset features (X_test)
X_test = test_df.copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (891, 7)
y shape: (891,)
X_test shape: (418, 7)


In [13]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Random Forest model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 2. Train the model on training data
model.fit(X, y)

# 3. Predict on test data
predictions = model.predict(X_test)

# Check predictions count
print("Predictions count:", len(predictions))
print("First 10 predictions:", predictions[:10])

Predictions count: 418
First 10 predictions: [0 0 0 0 0 0 1 0 1 0]


In [14]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

# Display individual fold scores and the average mean accuracy
print("Fold Accuracy Scores:", scores)
print(f"Mean Accuracy Score: {scores.mean() * 100:.2f}%")

Fold Accuracy Scores: [0.77653631 0.8258427  0.84831461 0.79213483 0.86516854]
Mean Accuracy Score: 82.16%


In [15]:
import pandas as pd

# 1. Create submission dataframe using saved PassengerId and predictions
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': predictions
})

# 2. Save dataframe as submission.csv
submission.to_csv('submission.csv', index=False)

# 3. Display first 5 rows of submission file
print("--- Submission File Created Successfully ---")
print(submission.head())

--- Submission File Created Successfully ---
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0


In [16]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Reload original datasets to get 'Name' back
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

# Save PassengerId for final submission
test_passenger_ids = test_df['PassengerId']

# Combine datasets for easier engineering
all_data = [train_df, test_df]

# --- FEATURE ENGINEERING ---

for dataset in all_data:
    # A. Fill missing values
    dataset['Age'].fillna(train_df['Age'].median(), inplace=True)
    dataset['Fare'].fillna(train_df['Fare'].median(), inplace=True)
    dataset['Embarked'].fillna('S', inplace=True)
    
    # B. Extract Title from Name
    dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    # Group rare titles into 'Rare' category
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    
    # Map Titles to numbers
    title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
    dataset['Title'] = dataset['Title'].map(title_mapping).fillna(0)
    
    # C. Create FamilySize and IsAlone features
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1
    
    # D. Encode categorical columns
    dataset['Sex'] = dataset['Sex'].map({'male': 0, 'female': 1})
    dataset['Embarked'] = dataset['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Drop unused columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

# Separate X and y
X = train_df.drop(columns=['Survived'])
y = train_df['Survived']
X_test = test_df.copy()

# --- MODEL TRAINING ---
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X, y)

# Predict on test data
predictions = model.predict(X_test)

# --- CREATE SUBMISSION ---
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)

print("--- Processing Completed & New submission.csv Created! ---")
print("Features used:", X.columns.tolist())

--- Processing Completed & New submission.csv Created! ---
Features used: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']
